# Updates

* 2025-03-20 updates
    * Chose fixed chunking with 250 words
        * SentenceTransformer model is limited to 384 tokens
        * Recursive chunker does not correctly return the split information (split overlap, source ID, page number)
    * Current tagging strategy
        * Ask LLM for one tag per chunk. Prompt v10 seems pretty good - go over prompt.
        * For each document, track the tagged and untagged chunks. Tags that show up in >= 30% (hyperparameter) of tagged chunks will be stored in a list. All tagged chunks will be tagged with this list.
        * Untagged chunks will be discarded
    * Current evaluation strategy
        * One label per chunk
        * Two metrics: found_label (recall), and number of tags (precision?)
    * Potential tagging strategy
        * Ask LLM for multiple tags per chunk. Prompt with example topics that fall into multiple categories, like how data cleaning can fall under both data science and data engineering, or how containerization can fall under both MLOps (MLE) and Infrastructure and Operations.
        * Otherwise, same as current strategy
    * Potential evaluation strategy
        * Multiple labels per chunk
        * Two metrics: recall, Jaccard similarity, precision?
* 2025-03-26 updates
    * PPTX conversion - actually, seems to work well for the most part (content is generally after title in the conversion, except for one exception)
    * Added multi-label tagging with tag averaging. Currently on prompt V14.
    * Added multi-label precision/recall/F1 through sklearn
    * Notebook conversion
* 2025-04-02 updates
    * Added file type to metadata for faster querying
    * Added default option to skip document cleaning in indexing pipeline - I think we want to keep formatting for the RAG system (new lines, headers, etc. indicate a structure that should help the LLM understand the content, both for tagging and generation)
    * First cut of labeling Jupyter notebooks and created train/test splits per notebook (80% train, 20% test)
* 2025-04-09
    * Performance on notebooks (all samples, no longer have data splits)
    * Lambda
        * One pipeline for indexing and tagging (show diagram)
        * First version of Lambda function (need to containerize it and test locally if possible, or on AWS if not)
* 2025-04-16
    * Completed deployment to AWS Lambda
    * Indexed and tagged all notebooks and pptx.
        * Notebook recall and precision are slightly down from before (not sure why). Recall: 92% -> 90.8%. Precision: 76.5% -> 75.1%.
        * PPTX recall = 97.9%, precision = 89.5%
    * First cut of documentation in README.md
* 2025-04-23
    * Made tagging options configurable via environment variables
    * Added deduplication component
    * Make sure that hf-xet warning is fixed
    * Scripts

# Notes

* Next steps
    * I passed a list of 430 training IDs into the tagging pipeline - why are only 429 documents written?
        * When I do not pass a list of IDs, there are 533 documents - why are only 532 written?
            * Only 532 results are retrieved from the queryer
            * That's because there's an ID collision because there are two copies of the same notebook: (Part 9 RAG/Langchain)/Demo_LangChain_2_Falcon7b_Instruct.ipynb
        * I passed in one file; worked fine
    * Calculate weighted recall, precision, F1 from CSVs
    * Docker + Lambda deployment
        <s>
        * Review Docker, Lambda
        * Clean up requirements.txt
        * Start with one Lambda function
            * Create a composite pipeline from the two pipelines (remove components and link them together if possible - or just add an option to insert the tagger into the indexing pipeline)
            * Update indexing pipeline - include option to add tagger into pipeline, add file type router, add joiner
            * Removing the file type context from the prompt significantly degrades per-class performance.
                * Calculate average performance
                * Can I modify the prompt on the fly depending on file type?
            * Test new indexing pipeline with full notebook dataset
        * Use idempotent code to create the collection
        * Move pipeline initialization inside lambda_handler (for now, until you figure out how to keep the HTTP connections alive)
        * v3 image: Run on all notebooks and check performance.
            * Performance is slightly worse with the new indexing pipeline/Lambda. Did something change in the indexing pipeline?
        * Documentation
        * Why is this warning being logged? `No abbreviations file found for en. Using default abbreviations.`
        * Why is this warning being logged in Cloudwatch? Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`. I installed hf-xet so this should be good.
        </s>
        * v4 image
            <s>
            * Add back check for haystack telemetry. Set env var in Docker image.
            * Make gpt model configurable via environment variable
            * Configurable untagged chunk option - I think you should generate new documents. I think all components generate new documents instead of modifying in place. Could be wrong. Embedder?
            </s>
            * Looks like if you hit the rate limit for generator, it will auto-retry? Initially I saw some logs that threw this exception. Double-check this.
            * Parse logs?
        * Error handling
            <s>
            * Early termination if unsupported file type (in lambda handler). Use mimetypes for now? Use filetype later? https://pypi.org/project/filetype/
            * What to do with untagged chunks
            </s>
            * LLM tagger response validation - hallucinating tags or format. Haystack validator/Pydantic.
            * What to do with ID collision - either skip or remove ID from vector store.
                * Interesting - when I wrote to the same collection 3 times, I got 22 * 3 loaded entities. In the data preview, I only saw 22. It looks like we still get 66 entries in the vector store.
                * Yes, it says 66 loaded entries even though I only get 22 results when I query
                * PrimaryKeyChecker component - after splitting? Check if ID between splitting and writing. ID does not change after splitting.
                * You might want to have this right before the writer. Otherwise, the race condition is worse because let's say you have two Lambda functions running, and one of them reaches DuplicateChecker before the other one writes to the vector database.
                * Other options: use Milvus upsert (is this atomic?), use S3/Redis for caching
            * Chunk length > max length - return error from Lambda? Lambda error handling? Unclassified document in file type router
        * Pydantic - field validator decorator, JSON schema validation, typeadapter. https://docs.pydantic.dev/latest/concepts/models/#basic-model-usage
        * From the S3 event, grab the bucket name and object key. Download the object to /tmp (ephemeral storage). For S3 -> Lambda, there will only be one object per invocation (each object created triggers its own Lambda), so you don't need much storage.
            * Technically, there can be multiple records per invocation, so you should write your code to loop through records. Later, if you want to batch to process multiple records per Lambda, you can use S3 -> EventBridge -> SQS -> Lambda. Or (S3 Inventory + scheduled Lambda for batch processing). Keep in mind that in this case, you would need to update your pipeline to handle all file formats (need some branching)
        * Later, if you want to use two functions (I don't think this is necessary), you can use AWS Step Functions or more simply, manually trigger the second function at the end of the first with boto3.invoke.
        * Can someone else (a different AWS account) trigger my Lambda function by uploading to their S3 bucket? If so, how do credentials work?
        * Do I really need to initialize my pipeline outside of the lambda handler? Is the same pipeline instance shared across warm invocations? Can this be a problem? 
            * How long does it take to initialize if the packages are already imported? It takes 2.5 seconds.
        * What is run each time Lambda is invoked? Everything in the file or just the handler? Importing packages?
        * If multiple Lambda invocations use the same warm container, then are the lambda_handlers processed synchronously? Then in this case would it make sense to initialize inside lambda_handler?
        * However, if multiple files are uploaded simultaneously while the container is cold, then multiple containers/execution environments are started?
        * Lambda warmup strategy?
        * Monitor CloudWatch logs to check pipeline performance. Monitor memory usage and /tmp capacity
        <s>
        * Error handling: fail early if files can't be downloaded or parsed
        * Clean up code if needed
        * I think you should get the written IDs from indexing pipeline
        * How to chain Lambda functions (probably one container for indexing, one container for tagging). AWS Step Functions.
        * Is there some sort of delay needed between inserting and querying? I think it should be okay - presumably Haystack only returns the number of documents written when Zilliz returns some confirmation message (check this)
            * The data insertion process may take some time to complete. It is recommended to wait a few seconds after inserting data and before conducting similarity searches. (https://docs.zilliz.com/docs/quick-start)
        </s>
    * Refine prompt for infra/ops if needed
    * Fine-tune prompt for Jupyter notebooks (training split only) - check tagging notes
        * Feed train IDs into tagging pipeline
        * pandas - update prompt to tag as DA, DE, DS, MLE.
    * Sanity-check the removed images in the Jupyter notebooks - there are some notebooks that explain how to write HTML, and that includes img tags.
    * Fix tagging for Metabase slides
    * Clean up the code
        * Pipeline for evaluation metric
        * Skip writing/reading to Zilliz; work locally since it should be faster
        * MLflow for experiment tracking
        * Refactor the code
* Labeling process
    * Initial labels
    * Tag
    * Check mismatched tags/labels (F1 score < 1) and see if labels need to be updated (at this point, do not worry about untagged chunks)
    * Once satisfied with the labels, split into train/test
* Labeling progress
    * IPYNB (with V14 prompt) - for now, update obvious mislabels. Then, fine-tune prompts and see if the tags improve (recall and precision). Be careful of adding too many labels - the labels should reflect the most important parts of the materials; they shouldn't seek to capture ALL possible labels. For example, if there's a little bit about Infra in slides about DS/MLE, that doesn't necessarily mean you should have the Infra label if that's not the real focus.
        * Data wrangling with Python - <u>this is all pandas tutorials, so I've updated the labels to DA, DE, DS, MLE.</u>
        * Part 10 LLM use cases
            * `Copy of 10.1 ES.ipynb`: Label = DS, MLE. Tag = Infra, DS, MLE. This one is about search and retrieval (Elasticsearch and vector search). <u>Do not update label.</u>
            * Label = DS, MLE. Tag = DE, DS, MLE. There's stuff in here about preprocessing data, but it's not REALLY about DE. <u>I think you should keep the labels as-is.</u>
                * `Time_Series/Copy of 1. RNN Exercise(EMG Time-Series Prediction) - Student.ipynb` 
                * `Time_Series/Copy of 1. RNN Exercise(EMG Time-Series Prediction) - Solution.ipynb`
                * `Time_Series/Copy of m3.1-data-prep-timeseries-ex2.ipynb`
                * `Time_Series/Copy of m3.1-data-prep-timeseries-lab1.ipynb`
                * ex1 and lab2 have not been tagged as DE, but they have some stuff about data preparation.
        * Part 3 data prep
            * `Data_Preparation/_m3.1-data-prep-text-lab-1.ipynb` - <u>this one is mistagged; do not update labels</u>
            * `Data_Preparation/Demo_LLM_Data_Prep_101.ipynb` - Label = DE, DS. Tag = DS, MLE. <u>Added MLE, check the unaveraged tags</u>
            * `Data_Collection/Demo_Web_Scraping_HTML_Xpath.ipynb` - Label = DE, DS. Tag = DA, DS, MLE. This one should be DE, DS. This notebook goes over how to write HTML and how to parse HTML. However, I think the LLM is confused because the writing HTML example is a data science portfolio, so it tags DA and MLE.
            * `Data_Preparation/Exercise_Solution_LLM_Data_Prep_Arxiv(WIP).ipynb` - tagged infra because of S3/boto code. But that's incidental imo. Do not update label.
            * Label = DE, DS. Tag = DE, DS, MLE
                * `Data_Preparation/Exercise_LLM_Data_Prep.ipynb`
                * `Data_Preparation/Exercise_Solution_LLM_Data_Prep.ipynb`
                * `Data_Preparation/Exercise_Solution_LLM_Data_Prep_for_Instruction_Tuning.ipynb` - <u>Added MLE to this label.</u>
            * `Data_Collection/Exercise_Solution_Web_Scraping_BS4.ipynb` - Label = DE, DS. Tag = DA, DE, DS. Do not update label. Notably the non-solution notebook is properly tagged.
            * `Data_Annotation/Exercise_Solution_LLM_Annotation_Sagemaker_Groundtruth.ipynb` - Label = DS, MLE. Tag = DE, DS, Infra, MLE. <u>This contains a lot of references to AWS services like S3 and Sagemaker, so added infra. It is distributed/cloud computing.</u>
        * Part 9 RAG - <u>there's a lot of DE tagged here, but PPTX are not tagged as DE. Is it because PPTX has less percentage of DE tag (tag averaging)? Or perhaps you need to more explicitly define that chunking and vector indexing is DS and MLE. Do not update tags for now</u>
            * `LangChain/Exercise_Solution_LangChain_6_ Build an Ask the Data app.ipynb` - this is mistagged. Probably add agents to DS/MLE tag.
            * `LangChain/LangChain_QA_Panel_App.ipynb` - this is also mistagged. Not sure about Infra tag.
            * `LlamaIndex/Demo_Llamaindex_Tutorial.ipynb` - mistagged
            * Label: DS, MLE. Tag: DE, DS, MLE. <u>Perhaps increase tag averaging instead of updating labels</u>
                * `Part_9_RAG/RAG/Demo_RAG_1_LangChain_FAISS.ipynb` - I think it considers chunking and indexing as part of DE. <u>Instead of updating label as DE, perhaps you should increase the tag averaging threshold for notebooks.</u>
                * `RAG/Exercise_Solution_RAG_2_ QA_with_LangChain.ipynb` (and the non-solution notebook) - <u>same as above</u>
                * `RAG/Exercise_Solution_RAG_5_with_Agents.ipynb` (and the non-solution notebook) - <u>same as above</u>
                * `RAG/(wip)_RAG_Index_Chunking_LangChain_Demo.ipynb` - <u>same as above</u>
                * `RAG/(wip)_RAG_Index_Retrieval_LlamaIndex_Demo.ipynb` - <u>same as above</u>
                * `RAG/Exercise_Solution_RAG_3_QA_with_LlamaIndex.ipynb` (and the non-solution notebook) - <u>there's stuff about chunking, embedding, vector store</u>
                * `RAG/Exercise_Solution_RAG_4_LlamaIndex_GPT_Finetuning.ipynb` - <u>there's some stuff in here - how come this isn't tagged as DE? Check the unaveraged tags. Although there's not THAT much here other than a line about indexing.</u>
            * `RAG/Exercise_Solution_RAG_1_LangChain_Chroma_Streamlit.ipynb` (and the non-solution notebook) - tag: DE, DS, MLE, infra. I think infra b/c streamlit deployment. <u>you might want to increase tag averaging instead of updating labels</u>
            * `RAG/Mini_Project_Deploy_and_Evaluate_Falcon_Model_for_RAG_with_AWS_Sagemaker/` - tags are (DS), (MLE), infra. Infra because it involves calling a model API endpoint? Perhaps do not update tags. <u>try increasing tag averaging</u>
* Tagging
    * I don't think it makes sense to have a validation set. 
        * Training - optimizing the parameters
        * Validation - optimizing the hyperparameters
        * I'm fine-tuning the prompt and the tag averaging. These are already hyperparameters - basically, all of the data is for validation because I'm not optimizing the parameters of the model.
        * However, we can still keep a holdout set for testing.
        * I don't think it makes sense to do train/test split for each document either. The tagging is averaged over documents. This is not a good way to evaluate generalization.
        * Check the prompts from the other team (job descriptions project)
    * Fine-tuning
        * V15 improves on V14; V15 and V17 are about the same.
        * V16 specifically calls out pandas, but I don't want to do this
        * Do not split into train/test - doesn't make sense. Tagging all IDs doesn't change performance too much.
        * Can reduce tag threshold and see how it goes
    * Prompts
        * V14 - not including SQL as an example topic means that some of the Metabase slides are untagged. Including document title did not help much.
        * Infra/ops - cloud computing is a subset of distributed computing. Double-check infra/ops for accuracy and make sure you understand the things that go into this category.
        * Consider adding common Python libraries (like pandas) to the prompt. What other libraries?
        * SQL, Python - I'd prefer not to call these out specifically, since it's really how you use SQL/Python that should determine the tag.
        * Should you include the document title? For now, I'd prefer not to because this could bias the model.
        * You may want to also include the file type in the prompt so the LLM knows what kind of format it's looking at
* Labeling
    * Using LLM to assist (you can check the slides): Ask LLM for label + confidence score before manual labeling
* Should I have a different pipeline where I prompt the LLM for their reasoning on the tagging? RAG pipeline. But temperature will be different? Can you use 0 for RAG?
* OCR for images?
* Perhaps you should clean up `<center>`, etc. See below. But this is actually about HTML (web scraping).

```
<center> </center> ---------- <h1 align="center"> Web Scraping 101 </h1>
<br/>
<center align="left"> <font size="4"> Developed by: </font><font color="#33AAFBD" size="4">WeCloudData</font></center>
<br/> ---------- # 1. Understanding HTML
HTML stands for `HyperText Markup Language`. Learn more about HTML: [w3schools css tutorials](https://www.w3schools.com/css/default.asp) ## Basic Terminology ## Rendering HTML in Jupyter In Jupyter Notebook, there're several ways of rendering (`display`) HTML file. > 1. IPython HTML module
> 2. %%html magic ### Render HTML in Jupyter with IPython HTML ```python
from IPython.core.display import HTML
HTML('''
<html> <head> <body> <h1>First Header</h1> <h2>Second Header</h2> <h3>Third Header</h3> </body> </head>
</html>
''')
``` ### Render HTML in Jupyter with `%%html` magic ```python
%%html <html> <head> <body> <h1>First Header</h1> <h2>Second Header</h2> <h3>Third Header</h3> </body> </head>
</html>
``` ## Basic Structure > - `<html>`
> - `<head>`
> - `<body>`
> - `<h1>, <h2>, <h3>`
> - `<a>`
> - ``</a></h3></h2></h1></body></head></html> ### Writing the basic structure ```python
%%html <html> <head> </head> <body> </body>
</html>
``` ### Adding Text ```python
%%html <html> <head> <title> Portfolio </title> </head> <body> <h1> My Data Science Portfolio </h1> <h2> by WeCloudData Academy </h2> <p> This web page contains the data science projects I worked during my learning journey. </p> </body>
</html>
``` ### Text Alignment ```python
%%html <html> <head> <title> Portfolio </title> </head> <body> <h1 align='center'> My Data Science Portfolio </h1> <h2 align='center'> by WeCloudData Academy </h2> <p> This web page contains the data science projects I worked during my learning journey. 

Demo_Web_Scraping_HTML_Xpath.ipynb
```

# Initialize environment and path

In [ ]:
import sys
from dotenv import load_dotenv
import os

load_dotenv(override=True)

if "../app/" not in sys.path:
    sys.path.append("../app/")

print(sys.path)
# print(os.getenv("OPENAI_API_KEY"))

In [ ]:
file_type = "ipynb"
file_ext = f".{file_type}"

# collection_name = "ipynb_skipclean_fixed250_tag30_prompt17_all_ids"
# collection_name = "ipynb_skipclean_fixed250_tag30_prompt18"
collection_name = "ryan_test_collection"
# collection_name = "ipynb_test"

# Download files

In [ ]:
import boto3, re

s3_client = boto3.client(
    "s3",
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    region_name=os.getenv("AWS_REGION"),
)

In [ ]:
bucket_name = "content-tagging-lms"

files = [d["Key"] for d in s3_client.list_objects(Bucket=bucket_name)["Contents"] if re.search(file_ext, d["Key"])]

for file in files:
    print(file)

# for file in files:
#     print(file)
#     head, _ = os.path.split(file)
#     print(head)
#     os.makedirs(head, exist_ok=True)
#     s3_client.download_file(bucket_name, file, file)

# Use pipeline to chunk

In [ ]:
from pathlib import Path

# files = list(Path(".").glob("Content/**/*.ipynb"))
files = list(Path(".").glob(f"Content/**/*{file_ext}"))
files

In [ ]:
from haystack_utilities.tools import create_collection

create_collection(collection_name, max_content_len_chars=65535)

In [ ]:
from haystack_utilities.pipelines import build_indexing_pipeline
from haystack_utilities.pipelines import indexing_pipeline_lambda
from haystack_utilities.tools import TaggingKwargs, DeduplicateOption

splitting_options = {
    "split_strategy": "recursive",
    "split_length": 100,
    "split_overlap": 20,
    "split_unit": "word",
    "separators": ["\f", "\n\n", "sentence", "\n", " "]
}

splitting_options = {
    "split_strategy": "recursive",
    "split_length": 200,
    "split_overlap": 40,
    "split_unit": "word",
    "separators": ["\f", "\n\n", "sentence", "\n", " "]
}

splitting_options = {
    "split_strategy": "recursive",
    "split_length": 300,
    "split_overlap": 60,
    "split_unit": "word",
    "separators": ["\f", "\n\n", "sentence", "\n", " "]
}

splitting_options = {
    "split_strategy": "fixed",
    "split_by": "word",
    "split_length": 300,
    "split_overlap": 60,
    "split_threshold": 300,
    "respect_sentence_boundary": True
}

splitting_options = {
    "split_strategy": "fixed",
    "split_by": "word",
    "split_length": 500,
    "split_overlap": 100,
    "split_threshold": 500,
    "respect_sentence_boundary": True
}

# sentence-transformers/all-mpnet-base-v2
# Max tokens = 384
# Roughly 288 words (3 words = 4 tokens)
# 'fixed250'
splitting_options = {
    "split_strategy": "fixed",
    "split_by": "word",
    "split_length": 250,
    "split_overlap": 50,
    "split_threshold": 30,
    "respect_sentence_boundary": True
}

# skip_cleaner=True
# index_pipe = build_indexing_pipeline(collection_name, splitting_options, skip_cleaner=skip_cleaner, file_extension=file_ext, max_content_len_chars=65535, drop_old=True)

skip_cleaner=True
add_tagger=True
add_tagger=False
tagging_kwargs = {
    "model": "gpt-4.1-mini-2025-04-14",
    "untagged_option": "discard",
    "temperature": "0.1",
    "max_completion_tokens": "40",
    "tag_threshold": "0.5",
}
# tagging_kwargs = {}
tagging_kwargs = TaggingKwargs(**tagging_kwargs)
dedup_option = DeduplicateOption(deduplicate_option="skip")
index_pipe = indexing_pipeline_lambda.build_indexing_pipeline(collection_name, splitting_options, skip_cleaner=skip_cleaner, add_tagger=add_tagger, tagging_kwargs=tagging_kwargs, deduplicate_option=dedup_option)

In [ ]:
# Remove the writer if you don't want to write to Zilliz Cloud

# index_pipe.remove_component("writer")
# index_pipe.remove_component("embedder")

index_pipe.show()

In [ ]:
fnames = files
fnames = ['Content/Data wrangling with Python/Class 7 - Advanced Pandas_Solutions.ipynb']
# fnames = ['Content/Data Visualization/4.2-Introduction to Metabase (BI).pptx']

# if not skip_cleaner:
#     index_results = index_pipe.run({"converter": {"sources": fnames}}, include_outputs_from={"converter", "cleaner", "metadata_cleaner"})
# else:
#     index_results = index_pipe.run({"converter": {"sources": fnames}}, include_outputs_from={"converter", "metadata_cleaner"})

if not skip_cleaner:
    index_results = index_pipe.run({"file_type_router": {"sources": fnames}}, include_outputs_from={"converter_document_joiner", "cleaner", "metadata_cleaner"})
else:
    index_results = index_pipe.run({"file_type_router": {"sources": fnames}}, include_outputs_from={"converter_document_joiner", "splitter", "metadata_cleaner", "tagger_document_joiner"})

print(len(index_results["metadata_cleaner"]["documents"]))
index_results

In [ ]:
set1 = set([doc.id for doc in index_results["splitter"]["documents"]])
set2 = set([doc.id for doc in index_results["tagger_document_joiner"]["documents"]])

In [ ]:
set1 == set2

In [ ]:
if "splitter" in index_results.keys():
    comp_name = "splitter"
elif "embedder" in index_results.keys():
    comp_name = "embedder"
elif "metadata_cleaner" in index_results.keys():
    comp_name = "metadata_cleaner"
else:
    comp_name = None

if comp_name:
    lengths = [len(doc.content) for doc in index_results[comp_name]["documents"]]
    print(f"Max len char = {max(lengths)}")
    print(f"Min len char = {min(lengths)}")

In [ ]:
import matplotlib.pyplot as plt

plt.hist(lengths)

In [ ]:
import re

# pattern = re.compile(r".ipynb$")
pattern = re.compile(f"{file_ext}$")
to_sub = r".md" if file_ext == ".ipynb" else r".txt"

os.makedirs(f"./pipe_outs/{file_type}", exist_ok=True)

component_names = ["converter", "cleaner"]
os.makedirs(f"./pipe_outs/{file_type}/converter", exist_ok=True)
os.makedirs(f"./pipe_outs/{file_type}/cleaner", exist_ok=True)

for component_name in component_names:
    if component_name in index_results.keys():
        for doc in index_results[component_name]["documents"]:
            filename = pattern.sub(to_sub, doc.meta["file_path"])
            full_filepath = f"./pipe_outs/{file_type}/{component_name}/{filename}"
            with open(full_filepath, "w") as file:
                file.write(doc.content)
                print(f"{full_filepath} written.")

In [ ]:
component_name = "metadata_cleaner"
os.makedirs(f"./pipe_outs/{file_type}/metadata_cleaner", exist_ok=True)

full_filepath = None
if component_name in index_results.keys():
    for doc in index_results[component_name]["documents"]:
        filename = pattern.sub(to_sub, doc.meta["metadata"]["title"])
        full_filepath_current = f"./pipe_outs/{file_type}/{component_name}/{filename}"
        if full_filepath == full_filepath_current:
            with open(full_filepath, "a") as file:
                file.write(f"{doc.content}\n\n{'<'*75}{'>'*75}\n\n")
        else:
            full_filepath = full_filepath_current
            with open(full_filepath, "w") as file:
                file.write(f"{doc.content}\n\n{'<'*75}{'>'*75}\n\n")

# Use pipeline to tag

In [ ]:
# Get training IDs (if available)
# import pandas as pd

# label_file = "ipynb_skipclean_fixed250_labels.pkl"
# df = pd.read_pickle(f"./labeling/{file_type}/{label_file}")

# training_ids = df.loc[df["split"] == "train"]["id"].to_list()
# print(len(training_ids))

In [ ]:
from haystack_utilities.pipelines import build_tagging_pipeline

# # V14 PPTX - iterating from PPTX V12
# prompt_template_tagging = """
#     I am doing a content tagging project. The goal of the project is to tag teaching materials.

#     I will give you a text chunk, delimited in triple backticks, and you will identify all applicable tags based on the text chunk. You can only pick tags from this list: Data Analysis, Data Engineering, Data Science, Infrastructure and Operations, Machine Learning Engineering.
    
#     Please format the tags as a Python list of strings. Respond only with the list, no other text, e.g. ["Machine Learning Engineering", "Data Science"].
    
#     If none of the tags apply, respond with an empty list, [].

#     Here are my guidelines for how to decide which tags to choose. I will give a topic followed by the recommended tags.
#     * Anything related to processing data before modeling or analysis, e.g. collecting, ingesting, cleaning, normalizing, and moving data: ["Data Engineering", "Data Science"].
#     * Anything related to database design and data modeling: ["Data Engineering", "Data Science"].
#     * Very simple data analysis like descriptive, diagnostic, and exploratory analysis, and data visualization: ["Data Analysis", "Data Science"].
#     * Business intelligence and Excel tutorials: ["Data Analysis"].
#     * Inferential data analysis: ["Data Science"].
#     * Feature engineering: ["Data Science"].
#     * Predictive and prescriptive data analysis: ["Data Science", "Machine Learning Engineering"].
#     * Statistical modeling techniques like linear regression, logistic regression, decision trees, SVM, etc.: ["Data Science", "Machine Learning Engineering"].
#     * Data labeling/annotation, both manual and with the help of LLMs: ["Data Science", "Machine Learning Engineering"].
#     * Traditional NLP techniques like TF-IDF, BM25, bag-of-words: ["Data Science", "Machine Learning Engineering"].
#     * Anything related to deep learning, including neural networks, modern computer vision, modern NLP, large language models, and AI: ["Data Science", "Machine Learning Engineering"].
#     * Anything related to retrieval-augmented generation (RAG), like vector indexing and vector search: ["Data Science", "Machine Learning Engineering"].
#     * Anything related to information retrieval and search engines: ["Data Science", "Machine Learning Engineering"].
#     * Anything related to deployment, operations, and scaling, including DevOps, CI/CD, cloud computing, computer networking, distributed computing, and containerization: ["Infrastructure and Operations"].
#     * Anything related to MLOps, model deployment, and model monitoring: ["Infrastructure and Operations", "Machine Learning Engineering"].

#     Text chunk: ```{{doc.content}}```
#     """

# V14 IPYNB - iterating from PPTX V14
prompt_template_tagging = """
    I am doing a content tagging project. The goal of the project is to tag teaching materials.

    I will give you a text chunk, delimited in triple backticks, and you will identify all applicable tags based on the text chunk. You can only pick tags from this list: Data Analysis, Data Engineering, Data Science, Infrastructure and Operations, Machine Learning Engineering.
    
    The text chunk is from a Jupyter Notebook that has been converted to Markdown.

    Please format the tags as a Python list of strings. Respond only with the list, no other text, e.g. ["Machine Learning Engineering", "Data Science"].
    
    If none of the tags apply, respond with an empty list, [].

    Here are my guidelines for how to decide which tags to choose. I will give a topic followed by the recommended tags.
    * Anything related to processing data before modeling or analysis, e.g. collecting, ingesting, cleaning, normalizing, and moving data: ["Data Engineering", "Data Science"].
    * Anything related to database design and data modeling: ["Data Engineering", "Data Science"].
    * Very simple data analysis like descriptive, diagnostic, and exploratory analysis, and data visualization: ["Data Analysis", "Data Science"].
    * Business intelligence and Excel tutorials: ["Data Analysis"].
    * Inferential data analysis: ["Data Science"].
    * Feature engineering: ["Data Science"].
    * Predictive and prescriptive data analysis: ["Data Science", "Machine Learning Engineering"].
    * Statistical modeling techniques like linear regression, logistic regression, decision trees, SVM, etc.: ["Data Science", "Machine Learning Engineering"].
    * Data labeling/annotation, both manual and with the help of LLMs: ["Data Science", "Machine Learning Engineering"].
    * Traditional NLP techniques like TF-IDF, BM25, bag-of-words: ["Data Science", "Machine Learning Engineering"].
    * Anything related to deep learning, including neural networks, modern computer vision, modern NLP, large language models, and AI: ["Data Science", "Machine Learning Engineering"].
    * Anything related to retrieval-augmented generation (RAG), like vector indexing and vector search: ["Data Science", "Machine Learning Engineering"].
    * Anything related to information retrieval and search engines: ["Data Science", "Machine Learning Engineering"].
    * Anything related to deployment, operations, and scaling, including DevOps, CI/CD, cloud computing, computer networking, distributed computing, and containerization: ["Infrastructure and Operations"].
    * Anything related to MLOps, model deployment, and model monitoring: ["Infrastructure and Operations", "Machine Learning Engineering"].

    Text chunk: ```{{doc.content}}```
    """

# V15 IPYNB - iterating from IPYNB V14. Added more notes on data preprocessing.
prompt_template_tagging = """
    I am doing a content tagging project. The goal of the project is to tag teaching materials.

    I will give you a text chunk, delimited in triple backticks, and you will identify all applicable tags based on the text chunk. You can only pick tags from this list: Data Analysis, Data Engineering, Data Science, Infrastructure and Operations, Machine Learning Engineering.
    
    The text chunk is from a Jupyter Notebook that has been converted to Markdown.

    Please format the tags as a Python list of strings. Respond only with the list, no other text, e.g. ["Machine Learning Engineering", "Data Science"].
    
    If none of the tags apply, respond with an empty list, [].

    Here are my guidelines for how to decide which tags to choose. I will give a topic followed by the recommended tags.
    * Anything related to processing data before modeling or analysis, e.g. collecting, ingesting, cleaning, normalizing, and moving data. This can include text preprocessing, image or video preprocessing, and any kind of data transformation. Examples of data collection include from databases, APIs, sensors, or via web scraping: ["Data Engineering", "Data Science"].
    * Anything related to database design and data modeling: ["Data Engineering", "Data Science"].
    * Very simple data analysis like descriptive, diagnostic, and exploratory analysis, and data visualization: ["Data Analysis", "Data Science"].
    * Business intelligence and Excel tutorials: ["Data Analysis"].
    * Inferential data analysis: ["Data Science"].
    * Feature engineering: ["Data Science"].
    * Predictive and prescriptive data analysis: ["Data Science", "Machine Learning Engineering"].
    * Statistical modeling techniques like linear regression, logistic regression, decision trees, SVM, etc.: ["Data Science", "Machine Learning Engineering"].
    * Data labeling/annotation, both manual and with the help of LLMs: ["Data Science", "Machine Learning Engineering"].
    * Traditional NLP techniques like TF-IDF, BM25, bag-of-words: ["Data Science", "Machine Learning Engineering"].
    * Anything related to deep learning, including neural networks, modern computer vision, modern NLP, large language models, and AI: ["Data Science", "Machine Learning Engineering"].
    * Anything related to retrieval-augmented generation (RAG), like vector indexing and vector search: ["Data Science", "Machine Learning Engineering"].
    * Anything related to information retrieval and search engines: ["Data Science", "Machine Learning Engineering"].
    * Anything related to deployment, operations, and scaling, including DevOps, CI/CD, cloud computing, computer networking, distributed computing, and containerization: ["Infrastructure and Operations"].
    * Anything related to MLOps, model deployment, and model monitoring: ["Infrastructure and Operations", "Machine Learning Engineering"].

    Text chunk: ```{{doc.content}}```
    """

# V16 IPYNB - iterating from IPYNB V15. Added pandas tutorial. I would rather not add anything too specific; the LLM tagger doesn't seem to understand "tutorial" all that well.
prompt_template_tagging = """
    I am doing a content tagging project. The goal of the project is to tag teaching materials.

    I will give you a text chunk, delimited in triple backticks, and you will identify all applicable tags based on the text chunk. You can only pick tags from this list: Data Analysis, Data Engineering, Data Science, Infrastructure and Operations, Machine Learning Engineering.
    
    The text chunk is from a Jupyter Notebook that has been converted to Markdown.

    Please format the tags as a Python list of strings. Respond only with the list, no other text, e.g. ["Machine Learning Engineering", "Data Science"].
    
    If none of the tags apply, respond with an empty list, [].

    Here are my guidelines for how to decide which tags to choose. I will give a topic followed by the recommended tags.
    * Anything related to processing data before modeling or analysis, e.g. collecting, ingesting, cleaning, normalizing, and moving data. This can include text preprocessing, image or video preprocessing, and any kind of data transformation. Examples of data collection include from databases, APIs, sensors, or via web scraping: ["Data Engineering", "Data Science"].
    * Anything related to database design and data modeling: ["Data Engineering", "Data Science"].
    * Very simple data analysis like descriptive, diagnostic, and exploratory analysis, and data visualization: ["Data Analysis", "Data Science"].
    * Business intelligence and Excel tutorials: ["Data Analysis"].
    * Pandas tutorials that are broadly useful across disciplines: ["Data Analysis", "Data Engineering", "Data Science", "Machine Learning Engineering"].
    * Inferential data analysis: ["Data Science"].
    * Feature engineering: ["Data Science"].
    * Predictive and prescriptive data analysis: ["Data Science", "Machine Learning Engineering"].
    * Statistical modeling techniques like linear regression, logistic regression, decision trees, SVM, etc.: ["Data Science", "Machine Learning Engineering"].
    * Data labeling/annotation, both manual and with the help of LLMs: ["Data Science", "Machine Learning Engineering"].
    * Traditional NLP techniques like TF-IDF, BM25, bag-of-words: ["Data Science", "Machine Learning Engineering"].
    * Anything related to deep learning, including neural networks, modern computer vision, modern NLP, large language models, and AI: ["Data Science", "Machine Learning Engineering"].
    * Anything related to retrieval-augmented generation (RAG), like vector indexing and vector search: ["Data Science", "Machine Learning Engineering"].
    * Anything related to information retrieval and search engines: ["Data Science", "Machine Learning Engineering"].
    * Anything related to deployment, operations, and scaling, including DevOps, CI/CD, cloud computing, computer networking, distributed computing, and containerization: ["Infrastructure and Operations"].
    * Anything related to MLOps, model deployment, and model monitoring: ["Infrastructure and Operations", "Machine Learning Engineering"].

    Text chunk: ```{{doc.content}}```
    """

# V17 IPYNB - iterating from IPYNB V15. Updated infra/ops prompts.
prompt_template_tagging = """
    I am doing a content tagging project. The goal of the project is to tag teaching materials.

    I will give you a text chunk, delimited in triple backticks, and you will identify all applicable tags based on the text chunk. You can only pick tags from this list: Data Analysis, Data Engineering, Data Science, Infrastructure and Operations, Machine Learning Engineering.
    
    The text chunk is from a Jupyter Notebook that has been converted to Markdown.

    Please format the tags as a Python list of strings. Respond only with the list, no other text, e.g. ["Machine Learning Engineering", "Data Science"].
    
    If none of the tags apply, respond with an empty list, [].

    Here are my guidelines for how to decide which tags to choose. I will give a topic followed by the recommended tags.
    * Anything related to processing data before modeling or analysis, e.g. collecting, ingesting, cleaning, normalizing, and moving data. This can include text preprocessing, image or video preprocessing, and any kind of data transformation. Examples of data collection include from databases, APIs, sensors, or via web scraping. Tag these as ["Data Engineering", "Data Science"].
    * Anything related to database design and data modeling: ["Data Engineering", "Data Science"].
    * Very simple data analysis like descriptive, diagnostic, and exploratory analysis, and data visualization: ["Data Analysis", "Data Science"].
    * Business intelligence and Excel tutorials: ["Data Analysis"].
    * Inferential data analysis: ["Data Science"].
    * Feature engineering: ["Data Science"].
    * Predictive and prescriptive data analysis: ["Data Science", "Machine Learning Engineering"].
    * Statistical modeling techniques like linear regression, logistic regression, decision trees, SVM, etc.: ["Data Science", "Machine Learning Engineering"].
    * Data labeling/annotation, both manual and with the help of LLMs: ["Data Science", "Machine Learning Engineering"].
    * Traditional NLP techniques like TF-IDF, BM25, bag-of-words: ["Data Science", "Machine Learning Engineering"].
    * Anything related to deep learning, including neural networks, modern computer vision, modern NLP, large language models, and AI: ["Data Science", "Machine Learning Engineering"].
    * Anything related to retrieval-augmented generation (RAG), like vector indexing and vector search: ["Data Science", "Machine Learning Engineering"].
    * Anything related to information retrieval and search engines: ["Data Science", "Machine Learning Engineering"].
    * Anything related to DevOps, including CI/CD, microservices, infrastructure as code (IaC), and application monitoring/logging. Anything related to deployment technologies, e.g. containerization, cloud services, and web application frameworks. Scalable computing infrastructure, e.g. distributed, parallel, and cloud computing. Tag these as ["Infrastructure and Operations"].
    * Anything related to MLOps, including distributed or cloud-based model training, model deployment, and post-deployment monitoring: ["Infrastructure and Operations", "Machine Learning Engineering"].

    Text chunk: ```{{doc.content}}```
    """

# V18 - iterating from V17 IPYNB. Removed note about Jupyter Notebook (prefer to use one prompt for all file types)
prompt_template_tagging = """
    I am doing a content tagging project. The goal of the project is to tag teaching materials.

    I will give you a text chunk, delimited in triple backticks, and you will identify all applicable tags based on the text chunk. You can only pick tags from this list: Data Analysis, Data Engineering, Data Science, Infrastructure and Operations, Machine Learning Engineering.

    Please format the tags as a Python list of strings. Respond only with the list, no other text, e.g. ["Machine Learning Engineering", "Data Science"].
    
    If none of the tags apply, respond with an empty list, [].

    Here are my guidelines for how to decide which tags to choose. I will give a topic followed by the recommended tags.
    * Anything related to processing data before modeling or analysis, e.g. collecting, ingesting, cleaning, normalizing, and moving data. This can include text preprocessing, image or video preprocessing, and any kind of data transformation. Examples of data collection include from databases, APIs, sensors, or via web scraping. Tag these as ["Data Engineering", "Data Science"].
    * Anything related to database design and data modeling: ["Data Engineering", "Data Science"].
    * Very simple data analysis like descriptive, diagnostic, and exploratory analysis, and data visualization: ["Data Analysis", "Data Science"].
    * Business intelligence and Excel tutorials: ["Data Analysis"].
    * Inferential data analysis: ["Data Science"].
    * Feature engineering: ["Data Science"].
    * Predictive and prescriptive data analysis: ["Data Science", "Machine Learning Engineering"].
    * Statistical modeling techniques like linear regression, logistic regression, decision trees, SVM, etc.: ["Data Science", "Machine Learning Engineering"].
    * Data labeling/annotation, both manual and with the help of LLMs: ["Data Science", "Machine Learning Engineering"].
    * Traditional NLP techniques like TF-IDF, BM25, bag-of-words: ["Data Science", "Machine Learning Engineering"].
    * Anything related to deep learning, including neural networks, modern computer vision, modern NLP, large language models, and AI: ["Data Science", "Machine Learning Engineering"].
    * Anything related to retrieval-augmented generation (RAG), like vector indexing and vector search: ["Data Science", "Machine Learning Engineering"].
    * Anything related to information retrieval and search engines: ["Data Science", "Machine Learning Engineering"].
    * Anything related to DevOps, including CI/CD, microservices, infrastructure as code (IaC), and application monitoring/logging. Anything related to deployment technologies, e.g. containerization, cloud services, and web application frameworks. Scalable computing infrastructure, e.g. distributed, parallel, and cloud computing. Tag these as ["Infrastructure and Operations"].
    * Anything related to MLOps, including distributed or cloud-based model training, model deployment, and post-deployment monitoring: ["Infrastructure and Operations", "Machine Learning Engineering"].

    Text chunk: ```{{doc.content}}```
    """

# AI agents? AI frameworks? Remove Excel?

tag_pipe = build_tagging_pipeline(collection_name, prompt_template_tagging)

In [ ]:
prompt_template_tagging

In [ ]:
tag_pipe.get_component("tagger").tag_pipe

In [ ]:
tag_results = tag_pipe.run({})
# tag_results = tag_pipe.run({"queryer": {"ids": training_ids}})

tag_results

# Check tagging results

In [ ]:
from pymilvus import MilvusClient
import os
from dotenv import load_dotenv

load_dotenv(override=True)

client = MilvusClient(
    uri=os.getenv("ZILLIZ_CLUSTER_ENDPOINT"),
    token=os.getenv("ZILLIZ_CLUSTER_TOKEN")
)

# collection_name = 'ryan_test_collection'
# os.makedirs(f"./experiments/{collection_name}", exist_ok=True)

In [ ]:
# client.release_collection("ryan_test_collection")
# client.load_collection("ryan_test_collection")
collection_name = "ryan_test_collection"

In [ ]:
tags = ["Machine Learning Engineering", "Data Analysis", "Data Engineering", "Data Science", "Infrastructure and Operations", "Other"]
# fname = 'Introduction to AWS.pptx'
# fname = "4.2-Introduction to Metabase (BI).pptx"

for tdx, tag in enumerate(tags):
    filter = f"json_contains(metadata['tags'], '{tag}')"
    # filter = f"json_contains(metadata['tags'], '{tag}') AND metadata['title'] == '{fname}'"
    res = client.query(
        collection_name=collection_name,
        filter=filter,
        output_fields=["text", "metadata"],
    )
    
    n_chunks = len(res)
    print(f"{tag}: {n_chunks}")

    # if os.path.exists(f"./experiments/tagged_chunks_{tdx}.txt"):
    #     os.remove(f"./experiments/tagged_chunks_{tdx}.txt")
    
    # with open(f"./experiments/tagged_chunks_{tdx}.txt", "a") as file:
    #     for r in res:
    #         # file.write(f"{r['text']}\n{"-"*100}\n\n")
    #         # file.write(r["metadata"]["title"])
    #         file.write(f"{r['text']}\n\n")
    #         file.write(f"{r['metadata']['title']}\n{"-"*100}\n")

    # fname = f"./experiments/{collection_name}/tagged_chunks_{tag}_{n_chunks}.txt"
    # if os.path.exists(fname):
    #     os.remove(fname)
    
    # with open(fname, "a") as file:
    #     for r in res:
    #         # file.write(f"{r['text']}\n{"-"*100}\n\n")
    #         # file.write(r["metadata"]["title"])
    #         file.write(f"{r['text']}\n\n")
    #         file.write(f"{r['metadata']['title']}\n{"-"*100}\n")

In [ ]:
filter = 'metadata["tags"] == []'
# filter = f"metadata['tags'] == [] AND metadata['title'] == '{fname}'"
res = client.query(
    collection_name=collection_name,
    filter=filter,
    output_fields=["text", "metadata"],
)

n_chunks = len(res)
print(n_chunks)

# fname = f"./experiments/{collection_name}/untagged_chunks_{n_chunks}.txt"
# if os.path.exists(fname):
#     os.remove(fname)

# with open(fname, "a") as file:
#     for r in res:
#         file.write(f"{r['text']}\n\n")
#         file.write(f"{r['metadata']['title']}\n{"-"*100}\n")

In [ ]:
lengths = [len(r['text']) for r in res]
print(lengths)

# Evaluate tagging accuracy

In [ ]:
import pandas as pd
import pickle

# Contains labels
# label_file = "df_labels_fixed250.pkl"
label_file = "ipynb_skipclean_fixed250_labels.pkl"

# collection_name = "pptx_lambda_collection"
# label_file = "pptx_lambda_collection_labels.pkl"
# file_type = "pptx"
# file_ext = ".pptx"
collection_name = "test_collection"
label_file = "ipynb_lambda_collection_labels.pkl"
file_type = "ipynb"
file_ext = ".ipynb"
has_tags = False
# label_file = "ipynb_skipclean_fixed250_tag30_prompt17_all_ids_tagged.pkl"
# has_tags = True
df = pd.read_pickle(f"./labeling/{file_type}/{label_file}")

client = MilvusClient(
    uri=os.getenv("ZILLIZ_CLUSTER_ENDPOINT"),
    token=os.getenv("ZILLIZ_CLUSTER_TOKEN"),
)

df.head(10)

In [ ]:
# Multilabel evaluation with sklearn metrics

import re
from sklearn.metrics import recall_score, precision_score, f1_score
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer

fname_base = f"{collection_name}_tagged"

classes=["Data Analysis", "Data Engineering", "Data Science", "Infrastructure and Operations", "Machine Learning Engineering"]
mlb = MultiLabelBinarizer(classes=classes)

d = {
    "id": [],
    "filename": [],
    # "split": [],
    "labels": [],
    "tags": [],
    "recall": [],
    "precision": [],
    "f1": [],
}

for idx in range(len(df)):
    # if df.loc[idx, "split"] == "test":
    #     continue
    
    if has_tags:
        tags = df.loc[idx, "tags"]
        labels = df.loc[idx, "labels"]
    else:
        res = client.get(
            collection_name=collection_name,
            ids=[df.loc[idx, "id"]],
            output_fields=["id", "metadata"]
        )

        if res == []:
            raise Exception("huh")
            continue
    
        tags = res[0]["metadata"]["tags"]

        labels = df.loc[idx, "label"]


        

    y_true = mlb.fit_transform([labels])
    y_pred = mlb.fit_transform([tags])

    recall = recall_score(y_true, y_pred, average="micro", zero_division=np.nan)
    precision = precision_score(y_true, y_pred, average="micro", zero_division=np.nan)
    f1 = f1_score(y_true, y_pred, average="micro", zero_division=np.nan)

    d["id"].append(df.loc[idx, "id"])
    d["filename"].append(df.loc[idx, "filename"])
    # d["split"].append(df.loc[idx, "split"])
    d["labels"].append(labels)
    d["tags"].append(tags)
    d["recall"].append(recall)
    d["precision"].append(precision)
    d["f1"].append(f1)
    # print(df.loc[idx, "id"])

# Aggregate metrics
y_true = mlb.fit_transform(d["labels"])
y_pred = mlb.fit_transform(d["tags"])
recall = recall_score(y_true, y_pred, average=None, zero_division=np.nan)
precision = precision_score(y_true, y_pred, average=None, zero_division=np.nan)
f1 = f1_score(y_true, y_pred, average=None, zero_division=np.nan)

recall_micro = recall_score(y_true, y_pred, average="micro", zero_division=np.nan)
precision_micro = precision_score(y_true, y_pred, average="micro", zero_division=np.nan)
f1_micro = f1_score(y_true, y_pred, average="micro", zero_division=np.nan)

recall_macro = recall_score(y_true, y_pred, average="macro", zero_division=np.nan)
precision_macro = precision_score(y_true, y_pred, average="macro", zero_division=np.nan)
f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=np.nan)

recall = np.append(recall, [recall_micro, recall_macro])
precision = np.append(precision, [precision_micro, precision_macro])
f1 = np.append(f1, [f1_micro, f1_macro])

with open(f"./labeling/{file_type}/{fname_base}.txt", "w") as file:
    width = 12
    classes_abbrev = ["DA", "DE", "DS", "InfraOps", "MLE", "AvgMicro", "AvgMacro"]
    file.write(f"{"Tag:":<{width}}" + "".join([f"{v:^{width}}" for v in classes_abbrev]) + "\n")
    file.write(f"{"Recall:":<{width}}" + "".join([f"{v:^{width}}" for v in np.round(recall,3)]) + "\n")
    file.write(f"{"Precision:":<{width}}" + "".join([f"{v:^{width}}" for v in np.round(precision,3)]) + "\n")
    file.write(f"{"F1:":<{width}}" + "".join([f"{v:^{width}}" for v in np.round(f1,3)]) + "\n")
    print(f"Saved ./labeling/{file_type}/{fname_base}.txt")

df_out = pd.DataFrame.from_dict(d)
df_out.to_pickle(f"./labeling/{file_type}/{fname_base}.pkl")
print(f"Saved ./labeling/{file_type}/{fname_base}.pkl")
df_out.to_csv(f"./labeling/{file_type}/{fname_base}.csv")
print(f"Saved ./labeling/{file_type}/{fname_base}.csv")
df_out

In [ ]:
collection_name = 'ipynb_skipclean_fixed250_tag30_prompt17'
fname_base = f"{collection_name}_tagged"


recall = [0.8630137,  0.81818182, 0.94186047, 1.,         0.95017794]
precision = [0.70786517, 0.30252101, 1.,         0.21621622, 0.85303514]
f1 = [0.77777778, 0.44171779, 0.97005988, 0.35555556, 0.8989899 ]


width = 12

classes = ["DA", "DE", "DS", "InfraOps", "MLE"]
print(f"{"Tag:":<{width}}" + "".join([f"{v:^{width}}" for v in classes]))
print(f"{"Recall:":<{width}}" + "".join([f"{v:^{width}}" for v in np.round(recall,3)]))
print(f"{"Precision:":<{width}}" + "".join([f"{v:^{width}}" for v in np.round(precision,3)]))
print(f"{"F1:":<{width}}" + "".join([f"{v:^{width}}" for v in np.round(f1,3)]))

with open(f"./labeling/{file_type}/{fname_base}.txt", "w") as file:
    file.write(f"{"Tag:":<{width}}" + "".join([f"{v:^{width}}" for v in classes]) + "\n")
    file.write(f"{"Recall:":<{width}}" + "".join([f"{v:^{width}}" for v in np.round(recall,3)]) + "\n")
    file.write(f"{"Precision:":<{width}}" + "".join([f"{v:^{width}}" for v in np.round(precision,3)]) + "\n")
    file.write(f"{"F1:":<{width}}" + "".join([f"{v:^{width}}" for v in np.round(f1,3)]) + "\n")
    print(f"Saved ./labeling/{file_type}/{fname_base}.txt")

In [ ]:
from pathlib import Path

results_dir = Path("./labeling/ipynb")
files = results_dir.glob("ipynb_skipclean_fixed250_tag30_prompt*.txt")
files = list(files)
files.sort()
files

In [ ]:
prompt_numbers = [14, 15, 16, 17, "17_all_ids", 18]

files = [Path(f"labeling/ipynb/ipynb_skipclean_fixed250_tag30_prompt{n}_tagged.txt") for n in prompt_numbers]
files

In [ ]:
import re

x_axis = []
results_dict = {} # tag -> metric -> list of values
for fdx, file in enumerate(files):
    with open(file, "r") as f:
        x = re.sub(r"ipynb_skipclean_fixed250_tag30_", "", file.name)
        x = re.sub(r"_tagged.txt", "", x)
        x_axis.append(x)

        results = f.read()
        results = re.sub(r":", "", results)
        results = re.split("\n", results)
        results = [r for r in results if r != ""]
        left_row = [re.match(r"\S+", r).group() for r in results[1:]]
        # print(left_row)
        header = re.findall(r"\S+", results[0])[1:]
        # print(header)
        metrics = [re.findall(r"\b[01]\.\d+", r) for r in results[1:]]
        metrics = [[float(m) for m in l] for l in metrics]
        # print(metrics)

        if fdx == 0:
            results_dict = {tag: {metric_name: [] for metric_name in left_row} for tag in header}

        for hdx, h in enumerate(header):
            for mdx, metric_name in enumerate(left_row):
                results_dict[h][metric_name].append(metrics[mdx][hdx])
        
results_dict


# results = re.sub(r":", "", results)
# results = re.split("\n", results)
# results = [r for r in results if r != ""]
# left_row = [re.match(r"\S+", r).group() for r in results[1:]]
# print(left_row)
# header = re.findall(r"\S+", results[0])[1:]
# print(header)
# metrics = [re.findall(r"\b[01]\.\d+", r) for r in results[1:]]
# metrics = [[float(m) for m in l] for l in metrics]
# print(metrics)

# results_dict = {tag: {metric_name: [] for metric_name in left_row} for tag in header}
# for hdx, h in enumerate(header):
#     for mdx, metric_name in enumerate(left_row):
#         results_dict[h][metric_name].append(metrics[mdx][hdx])
# print(results_dict)



# results
# re.sub("\s+", " ", results)
# results

# metrics = re.findall(r"\b[01]\.\d+\b", results)
# metrics = [float(m) for m in metrics]
# metrics

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(len(results_dict.keys()), 1, figsize=(10,40))
for tdx, tag in enumerate(results_dict.keys()):
    ax = axs[tdx]
    for metric in results_dict[tag].keys():
        ax.plot(results_dict[tag][metric], label=metric)
    
    ax.legend(loc="upper left")
    ax.set_xticks(list(range(len(x_axis))))
    ax.set_xticklabels(x_axis)
    ax.grid()
    ax.set_title(tag)

# Averaging tags per document

For each document, keep track of how many times each tag shows up. For tags that meet some minimum threshold (e.g. >= 30% of tagged chunks), keep them and tag all chunks with that tag. Otherwise, discard. Untagged chunks remain untagged.

In [ ]:
# collection_name = "pptx_fixed250_prompt14"

client = MilvusClient(
    uri=os.getenv("ZILLIZ_CLUSTER_ENDPOINT"),
    token=os.getenv("ZILLIZ_CLUSTER_TOKEN")
)

# Contains labels
label_file = "df_labels_fixed250.pkl"
df = pd.read_pickle(f"./labeling/pptx/{label_file}")

client = MilvusClient(
    uri=os.getenv("ZILLIZ_CLUSTER_ENDPOINT"),
    token=os.getenv("ZILLIZ_CLUSTER_TOKEN"),
)

In [ ]:
threshold = 30
fname = f"./labeling/pptx/dict_averaged_tags_fixed250_tag{threshold}_prompt14.pkl"

dict_averaged_tags = {}

doc_dict = {}
for idx in range(len(df)):
    doc_id = df.loc[idx, "id"]
    # print(doc_id)
    res = client.get(
        collection_name=collection_name,
        ids=[doc_id],
        output_fields=["id", "metadata"]
    )

    source_id = res[0]["metadata"]["source_id"]

    if source_id not in doc_dict:
        doc_dict[source_id] = {
            "tagged_docs": [],
            "untagged_docs": [],
            "tags": {}
        }

    tags = res[0]["metadata"]["tags"]

    if not isinstance(tags, list):
        raise Exception(f"SyncLLMTagger.run(): Expected tags to be list, but got {type(tags)}")
    elif tags == []:
        # doc_dict[source_id]["untagged_docs"].append(doc_id)
        dict_averaged_tags[doc_id] = []
    else:
        doc_dict[source_id]["tagged_docs"].append(doc_id)
        for tag in tags:
            doc_dict[source_id]["tags"][tag] = doc_dict[source_id]["tags"].get(tag, 0) + 1

for source_id in doc_dict:
    total = len(doc_dict[source_id]["tagged_docs"])
    if total > 0:
        keep_tags = [tag for tag, count in doc_dict[source_id]["tags"].items() if count/total >= threshold/100]
        for doc in doc_dict[source_id]["tagged_docs"]:
            dict_averaged_tags[doc] = keep_tags



with open(fname, 'wb') as file:
    pickle.dump(dict_averaged_tags, file)

dict_averaged_tags

# Visualize experiments

# Sanity-check retrieval and generation

In [ ]:
from haystack_utilities.pipelines import build_rag_pipeline

prompt_rag = """
You will be provided some context, followed by the page number that this context comes from.
Answer the question based on the contexts, and reference the page numbers from which your answer is generated.

Context:
{% for doc in documents %}
   {{ doc.content }} 
   Page: {{ doc.meta['metadata']["page_number"]}}
   -----
{% endfor %}
-----
Question: {{ query }}
-----
Answer:
"""

# prompt_rag = """
# Please answer the question. Ignore the context provided.

# Context:
# {% for doc in documents %}
#    {{ doc.content }} 
#    Page: {{ doc.meta['metadata']["page_number"]}}
#    -----
# {% endfor %}
# -----
# Question: {{ query }}
# -----
# Answer:
# """

rag_pipe = build_rag_pipeline(collection_name, prompt_rag, top_k=3)

In [ ]:
query = "How can I use LLMs to assist in labeling datasets?"
query = "Please provide a detailed step-by-step instruction for labeling a dataset with LLM assistance, e.g. Step 1, Step 2, Step 3, etc."
# query = "How many teams are there in the NBA?"

In [ ]:
rag_results = rag_pipe.run({"query_embedder": {"text": query}, "prompt_builder": {"query": query}}, include_outputs_from={"retriever", "prompt_builder"})

rag_results

In [ ]:
if "prompt_builder" in rag_results:
    print(rag_results["prompt_builder"]["prompt"])

In [ ]:
if not os.path.exists("./experiments"):
    os.mkdir("./experiments")
        
fname = f"./experiments/generator_output.txt"
with open(fname, "w") as file:
    file.write(rag_results["generator"]["replies"][0])
    print(f"{fname} written.")


component_names = ["retriever"]

for name in component_names:
    if name in rag_results.keys():
        fname = f"./experiments/{name}_output.txt"

        if os.path.exists(fname):
            os.remove(fname)

        with open(f"./experiments/{name}_output.txt", "a") as file:
            for doc in rag_results[name]["documents"]:
                file.write(f"{doc.content}\n{"-"*200}\n")
            print(f"{fname} written.")

# Drop collection

In [ ]:
from haystack_utilities.tools import MilvusContextManager

with MilvusContextManager() as client:
    print(client.list_collections())
    # client.drop_collection('pptx_fixed250_tag30_prompt14')
    # client.drop_collection('ipynb_skipclean_fixed250_tag30_prompt17_all_ids')
    # client.drop_collection('ipynb_skipclean_fixed250_tag30_prompt18')
    client.drop_collection("ryan_test_collection")
    # client.drop_collection("lambda_test_collection")
    # client.drop_collection("pptx_lambda_collection")
    # client.drop_collection("ipynb_lambda_collection")
    print(client.list_collections())

# client.list_collections()
# client.drop_collection('pptx_check_conversion')

# Chunk lengths

In [ ]:
from pathlib import Path

files = list(Path(".").glob("./labeling/ipynb/lambda_collection/*"))
files = list(Path(".").glob("./labeling/pptx/lambda_collection/*"))

lengths = []
for file in files:
    with open(file, "r") as f:
        txt = f.read()
        lengths.append(len(txt))

In [ ]:
import matplotlib.pyplot as plt

plt.hist(lengths)
plt.grid()
plt.xlabel("Chunk Length (Characters)")
plt.ylabel("Number of Chunks")
# plt.title("Jupyter Notebooks")
plt.title("Powerpoints")

# Old tagging prompts

In [ ]:
# V1
prompt_template_tagging = """
    I am doing a content tagging project. The goal of the project is to tag teaching materials.

    Tags: [Machine Learning Engineering, Data Analysis, Data Engineering, Data Science, Other].

    Here are some examples of topics for each tag:
    * Machine Learning Engineering: AI, Generative AI, Natural Language Processing, Large Language Models, Computer Vision, MLOps
    * Data Science: Statistics, Predictive Modeling, Feature Engineering, Data Science
    * Data Analysis: Data Analytics, Business Intelligence, SQL, Excel, Visualization
    * Data Engineering: Data Pipelines, Big Data, Cloud Engineering, DevOps, Spark
    * Other: Web Scraping

    I will give you a text chunk, delimited in triple backticks, and you will identify the most likely tag based on the text chunk.

    Please format the tag as a string inside a 1-element Python list, e.g. ["Machine Learning Engineering"].
    
    Respond only with the list, no other text.

    If none of the tags apply, respond with an empty list, [].

    Text chunk: ```{{doc.content}}```
    """

# V2
prompt_template_tagging = """
    I am doing a content tagging project. The goal of the project is to tag teaching materials.

    Tags: [Machine Learning Engineering, Data Analysis, Data Engineering, Data Science, Infrastructure and Operations].

    Here are some examples of topics for each tag:
    * Machine Learning Engineering:
        - Fine-Tuning, Parameter-Efficient Fine-Tuning (PEFT), Low-Rank Adaptation (LoRA)
        - AI, Generative AI (GenAI), AI Agent
        - Natural Language Processing (NLP), Large Language Models (LLM), Prompt Engineering
        - Retrieval-Augmented Generation (RAG), Vector Databases, Vector Search, Chunking
        - Computer Vision
    * Data Science:
        - Statistical Modeling, Statistical Analysis, Statistics
        - Feature Engineering
        - Classic Information Retrieval, BM25, TF-IDF, Bag of Words
    * Data Analysis:
        - Business Intelligence (BI)
        - SQL, Excel
        - Exploratory Data Analysis (EDA)
        - Data Visualization, Reporting
    * Data Engineering:
        - ETL, Data Pipelines, Data Collection, Data Cleaning, Web Scraping
        - Big Data, Spark, Hadoop, Kafka
    * Infrastructure and Operations:
        - MLOps, Model Deployment
        - DevOps, CI/CD
        - Cloud, AWS, GCP, Azure
        - Distributed Systems
        - Networking

    I will give you a text chunk, delimited in triple backticks, and you will identify the most likely tag based on the text chunk.

    Please format the tag as a string inside a 1-element Python list, e.g. ["Machine Learning Engineering"].
    
    Respond only with the list, no other text.

    If none of the tags apply, respond with an empty list, [].

    Text chunk: ```{{doc.content}}```
    """

# V3
prompt_template_tagging = """
    I am doing a content tagging project. The goal of the project is to tag teaching materials.

    Tags: [Machine Learning Engineering, Data Analysis, Data Engineering, Data Science, Infrastructure and Operations].

    Here are some examples of topics for each tag:
    * Machine Learning Engineering: Machine Learning, Fine-Tuning, Artificial Intelligence (AI), Natural Language Processing (NLP), Large Language Models (LLM), Prompt Engineering, Retrieval-Augmented Generation (RAG), Computer Vision
    * Data Science: Data Science, Statistical Analysis, Statistical Modeling, Feature Engineering
    * Data Analysis: Data Analysis, Analytics, Business Intelligence (BI), SQL, Excel, Data Visualization
    * Data Engineering: Data Engineering, ETL, Data Wrangling, Data Pipelines, Big Data (Spark, Hadoop, Kafka), Data Storage, Web Scraping
    * Infrastructure and Operations: MLOps, Model Deployment, Model Monitoring, DevOps, CI/CD, Cloud (AWS, GCP, Azure), Networks

    I will give you a text chunk, delimited in triple backticks, and you will identify the most likely tag based on the text chunk.

    Please format the tag as a string inside a 1-element Python list, e.g. ["Machine Learning Engineering"].
    
    Respond only with the list, no other text.

    If none of the tags apply, respond with an empty list, [].

    Text chunk: ```{{doc.content}}```
    """

# V4
prompt_template_tagging = """
    I am doing a content tagging project. The goal of the project is to tag teaching materials.

    Tags: [Machine Learning Engineering, Data Analysis, Data Engineering, Data Science, Infrastructure and Operations].

    Here are some examples of topics for each tag:
    * Machine Learning Engineering: AI, Natural Language Processing, Large Language Models, Prompt Engineering, Retrieval-Augmented Generation, Similarity Search, Vector Search, Vector Index, Computer Vision
    * Data Science: Statistical Analysis, Statistical Modeling, Feature Engineering, Data Annotation, Data Labeling
    * Data Analysis: Data Analytics, Business Intelligence, SQL, Excel, Data Visualization
    * Data Engineering: ETL, Data Wrangling, Deduplication, Data Pipelines, Big Data (Spark, Hadoop, Kafka), Data Storage, Web Scraping
    * Infrastructure and Operations: MLOps, Model Deployment, Model Monitoring, DevOps, CI/CD, Cloud (AWS, GCP, Azure), Networks

    I will give you a text chunk, delimited in triple backticks, and you will identify the most likely tag based on the text chunk.

    Please format the tag as a string inside a 1-element Python list, e.g. ["Machine Learning Engineering"].
    
    Respond only with the list, no other text.

    If none of the tags apply, respond with an empty list, [].

    Text chunk: ```{{doc.content}}```
    """

# V5
prompt_template_tagging = """
    I am doing a content tagging project. The goal of the project is to tag teaching materials.

    Tags: [Machine Learning Engineering, Data Analysis, Data Engineering, Data Science, Infrastructure and Operations].

    Here are some examples of topics for each tag:
    * Machine Learning Engineering: AI, Natural Language Processing, Large Language Models, Prompt Engineering, Retrieval-Augmented Generation, Similarity Search, Vector Search, Vector Index, Computer Vision
    * Data Science: Statistical Analysis, Statistical Modeling, Feature Engineering, Data Annotation, Data Labeling
    * Data Analysis: Data Analytics, Business Intelligence, SQL, Excel, Data Visualization
    * Data Engineering: ETL, Data Wrangling, Deduplication, Data Pipelines, Big Data (Spark, Hadoop, Kafka), Data Storage, Web Scraping
    * Infrastructure and Operations: MLOps, Model Deployment, Model Monitoring, DevOps, CI/CD, Cloud (AWS, GCP, Azure), Computer Networks, Distributed Computing

    I will give you a text chunk, delimited in triple backticks, and you will identify the most likely tag based on the text chunk.

    Please format the tag as a string inside a 1-element Python list, e.g. ["Machine Learning Engineering"].
    
    Respond only with the list, no other text.

    If none of the tags apply, respond with an empty list, [].

    Text chunk: ```{{doc.content}}```
    """

# V6
prompt_template_tagging = """
    I am doing a content tagging project. The goal of the project is to tag teaching materials.

    Tags: [Machine Learning Engineering, Data Analysis, Data Engineering, Data Science, Infrastructure and Operations].

    Here are some examples of topics for each tag:
    * Machine Learning Engineering: AI, Natural Language Processing, Large Language Models, Prompt Engineering, Retrieval-Augmented Generation, Similarity Search, Vector Search, Vector Index, Computer Vision
    * Data Science: Statistical Analysis, Statistical Modeling, Feature Engineering, Data Annotation, Data Labeling, Information Retrieval (TF-IDF, BM25, Bag-of-Words)
    * Data Analysis: Data Analytics, Business Intelligence, SQL, Excel, Data Visualization
    * Data Engineering: ETL (Extract Transform Load), Data Collection, Data Wrangling, Data Pipelines, Big Data (Spark, Hadoop, Kafka), Data Storage
    * Infrastructure and Operations: MLOps, Model Deployment, Model Monitoring, DevOps, CI/CD, Cloud (AWS, GCP, Azure), Computer Networks, Distributed Computing

    I will give you a text chunk, delimited in triple backticks, and you will identify the most likely tag based on the text chunk.

    Please format the tag as a string inside a 1-element Python list, e.g. ["Machine Learning Engineering"].
    
    Respond only with the list, no other text.

    If none of the tags apply, respond with an empty list, [].

    Text chunk: ```{{doc.content}}```
    """

# V7
# I'm choosing to put Information Retrieval under ML since so many information retrieval systems and search engines are using or trying to use ML, especially since all of these fields are about data.
prompt_template_tagging = """
    I am doing a content tagging project. The goal of the project is to tag teaching materials.

    Tags: [Machine Learning Engineering, Data Analysis, Data Engineering, Data Science, Infrastructure and Operations].

    Here are some examples of topics for each tag:
    * Data Engineering: Data Pipelines, Data Ingestion, Data Transformation (Cleaning, Standardizing, Aggregation, Enrichment, Filtering), Data Storage (Database Design, Data Modeling), ETL, ELT
    * Data Analysis: Basic Data Analysis (Descriptive, Diagnostic, Exploratory), Business Intelligence, Data Visualization (Dashboards), SQL, Excel
    * Data Science: Advanced Data Analysis (Inferential, Predictive, Prescriptive), Statistical Modeling, Feature Engineering, Data Labeling/Annotation
    * Machine Learning Engineering: Deep Learning, Neural Networks, AI, Computer Vision, Natural Language Processing, Large Language Models, Retrieval-Augmented Generation, Information Retrieval
    * Infrastructure and Operations: DevOps, CI/CD, Cloud Computing, Computer Networks, Distributed Computing, Containerization, MLOps, Model Deployment, Model Monitoring

    I will give you a text chunk, delimited in triple backticks, and you will identify the most likely tag based on the text chunk.

    Please format the tag as a string inside a 1-element Python list, e.g. ["Machine Learning Engineering"].
    
    Respond only with the list, no other text.

    If none of the tags apply, respond with an empty list, [].

    Text chunk: ```{{doc.content}}```
    """

# V8
prompt_template_tagging = """
    I am doing a content tagging project. The goal of the project is to tag teaching materials.

    Tags: [Machine Learning Engineering, Data Analysis, Data Engineering, Data Science, Infrastructure and Operations].

    I will give you a text chunk, delimited in triple backticks, and you will identify the most likely tag based on the text chunk.

    Here are my guidelines for how to decide which tag to choose.
    1. Data Engineering: Anything related to the automated processing of data prior to being used in modeling or analysis. Collecting, ingesting, cleaning, normalizing, and moving data. Anything related to database design and data modeling.
    2. Data Analysis: Very simple data analysis like descriptive, diagnostic, and exploratory analysis, and data visualization like dashboards. Other keywords: business intelligence, SQL, Excel.
    3. Data Science: Advanced data analysis like inferential, predictive, and prescriptive analysis. Statistical modeling techniques like linear regression, logistic regression, decision trees, SVM, etc. Traditional NLP techniques like TF-IDF, BM25, bag-of-words. Other keywords: Feature engineering, data labeling/annotation.
    4. Machine Learning Engineering: Anything related to deep learning, including neural networks, modern computer vision, modern NLP, large language models, retrieval-augmented generation, AI.
    5. Infrastructure and Operations: Anything related to deployment, operations, and scaling, including DevOps, CI/CD, cloud computing, computer networking, distributed computing, containerization, MLOps, model deployment, and model monitoring.

    Please format the tag as a string inside a 1-element Python list, e.g. ["Machine Learning Engineering"].
    
    Respond only with the list, no other text.

    If none of the tags apply, respond with an empty list, [].

    Text chunk: ```{{doc.content}}```
    """

# V9
prompt_template_tagging = """
    I am doing a content tagging project. The goal of the project is to tag teaching materials.

    Tags: [Machine Learning Engineering, Data Analysis, Data Engineering, Data Science, Infrastructure and Operations].

    I will give you a text chunk, delimited in triple backticks, and you will identify the most likely tag based on the text chunk.

    Here are my guidelines for how to decide which tag to choose.
    1. Data Engineering: Anything related to the automated processing of data prior to being used in modeling or analysis. Collecting, ingesting, cleaning, normalizing, and moving data. Anything related to database design and data modeling.
    2. Data Analysis: Very simple data analysis like descriptive, diagnostic, and exploratory analysis, and data visualization like dashboards. Other keywords: business intelligence, SQL, Excel.
    3. Data Science: Advanced data analysis like inferential, predictive, and prescriptive analysis. Statistical modeling techniques like linear regression, logistic regression, decision trees, SVM, etc. Data labeling/annotation, both manual and with the help of LLMs. Traditional NLP techniques like TF-IDF, BM25, bag-of-words. Other keywords: Feature engineering, time series analysis.
    4. Machine Learning Engineering: Anything related to deep learning, including neural networks, modern computer vision, modern NLP, large language models, AI, and datasets for training LLMs. Anything related to retrieval-augmented generation (RAG), like vector indexing and vector search. Anything related to information retrieval and search engines.
    5. Infrastructure and Operations: Anything related to deployment, operations, and scaling, including DevOps, CI/CD, cloud computing, computer networking, distributed computing, containerization, MLOps, model deployment, and model monitoring.

    Please format the tag as a string inside a 1-element Python list, e.g. ["Machine Learning Engineering"].
    
    Respond only with the list, no other text.

    If none of the tags apply, respond with an empty list, [].

    Text chunk: ```{{doc.content}}```
    """

# V10
prompt_template_tagging = """
    I am doing a content tagging project. The goal of the project is to tag teaching materials.

    Tags: [Machine Learning Engineering, Data Analysis, Data Engineering, Data Science, Infrastructure and Operations].

    I will give you a text chunk, delimited in triple backticks, and you will identify the most likely tag based on the text chunk.

    Here are my guidelines for how to decide which tag to choose.
    1. Data Engineering: Anything related to the automated processing of data prior to being used in modeling or analysis. Collecting, ingesting, cleaning, normalizing, and moving data. Anything related to database design and data modeling.
    2. Data Analysis: Very simple data analysis like descriptive, diagnostic, and exploratory analysis, and data visualization like dashboards. Other keywords: business intelligence, SQL, Excel.
    3. Data Science: Advanced data analysis like inferential, predictive, and prescriptive analysis. Statistical modeling techniques like linear regression, logistic regression, decision trees, SVM, etc. Data labeling/annotation, both manual and with the help of LLMs. Traditional NLP techniques like TF-IDF, BM25, bag-of-words. Other keywords: Feature engineering, time series analysis.
    4. Machine Learning Engineering: Anything related to deep learning, including neural networks, modern computer vision, modern NLP, large language models, AI. Anything related to retrieval-augmented generation (RAG), like vector indexing and vector search. Anything related to information retrieval and search engines.
    5. Infrastructure and Operations: Anything related to deployment, operations, and scaling, including DevOps, CI/CD, cloud computing, computer networking, distributed computing, containerization, MLOps, model deployment, and model monitoring.

    Please format the tag as a string inside a 1-element Python list, e.g. ["Machine Learning Engineering"].
    
    Respond only with the list, no other text.

    If none of the tags apply, respond with an empty list, [].

    Text chunk: ```{{doc.content}}```
    """

# V11 - multi-label tagging
prompt_template_tagging = """
    I am doing a content tagging project. The goal of the project is to tag teaching materials.

    Tags: [Data Analysis, Data Engineering, Data Science, Infrastructure and Operations, Machine Learning Engineering].

    I will give you a text chunk, delimited in triple backticks, and you will identify all applicable tags based on the text chunk.
    
    Please format the tags as a Python list of strings, e.g. ["Machine Learning Engineering", "Data Science"].
    
    Respond only with the list, no other text. If none of the tags apply, respond with an empty list, [].

    Here are my guidelines for how to decide which tag to choose.
    * Anything related to processing data before modeling or analysis, e.g. collecting, ingesting, cleaning, normalizing, and moving data, should be tagged as ["Data Engineering", "Data Science"].
    * Anything related to database design and data modeling should be tagged as ["Data Engineering", "Data Science"].
    * Very simple data analysis like descriptive, diagnostic, and exploratory analysis, and data visualization should be tagged as ["Data Analysis", "Data Science"].
    * Business intelligence and Excel tutorials should be tagged as ["Data Analysis"].
    * SQL tutorials should be tagged as ["Data Analysis", "Data Engineering", "Data Science", "Machine Learning Engineering"].
    * Inferential data analysis should be tagged as ["Data Science"].
    * Feature engineering should be tagged as ["Data Science"].
    * Predictive and prescriptive data analysis should be tagged as ["Data Science", "Machine Learning Engineering"].
    * Statistical modeling techniques like linear regression, logistic regression, decision trees, SVM, etc. should be tagged as ["Data Science", "Machine Learning Engineering"].
    * Data labeling/annotation, both manual and with the help of LLMs, should be tagged as ["Data Science", "Machine Learning Engineering"].
    * Traditional NLP techniques like TF-IDF, BM25, bag-of-words should be tagged as ["Data Science", "Machine Learning Engineering"].
    * Anything related to deep learning, including neural networks, modern computer vision, modern NLP, large language models, and AI should be tagged as ["Data Science", "Machine Learning Engineering"].
    * Anything related to retrieval-augmented generation (RAG), like vector indexing and vector search, should be tagged as ["Data Science", "Machine Learning Engineering"].
    * Anything related to information retrieval and search engines should be tagged as ["Data Science", "Machine Learning Engineering"].
    * Anything related to deployment, operations, and scaling, including DevOps, CI/CD, cloud computing, computer networking, distributed computing, and containerization, should be tagged as ["Infrastructure and Operations"].
    * Anything related to MLOps, model deployment, and model monitoring should be tagged as ["Infrastructure and Operations", "Machine Learning Engineering"].

    Text chunk: ```{{doc.content}}```
    """

# V12
prompt_template_tagging = """
    I am doing a content tagging project. The goal of the project is to tag teaching materials.

    Tags: Data Analysis, Data Engineering, Data Science, Infrastructure and Operations, Machine Learning Engineering.

    I will give you a text chunk, delimited in triple backticks, and you will identify all applicable tags based on the text chunk.
    
    Please format the tags as a Python list of strings. Respond only with the list, no other text, e.g. ["Machine Learning Engineering", "Data Science"].
    
    If none of the tags apply, respond with an empty list, [].

    Here are my guidelines for how to decide which tags to choose. I will give a topic followed by the recommended tags.
    * Anything related to processing data before modeling or analysis, e.g. collecting, ingesting, cleaning, normalizing, and moving data: ["Data Engineering", "Data Science"].
    * Anything related to database design and data modeling: ["Data Engineering", "Data Science"].
    * Very simple data analysis like descriptive, diagnostic, and exploratory analysis, and data visualization: ["Data Analysis", "Data Science"].
    * Business intelligence and Excel tutorials: ["Data Analysis"].
    * SQL tutorials: ["Data Analysis", "Data Engineering", "Data Science", "Machine Learning Engineering"].
    * Inferential data analysis: ["Data Science"].
    * Feature engineering: ["Data Science"].
    * Predictive and prescriptive data analysis: ["Data Science", "Machine Learning Engineering"].
    * Statistical modeling techniques like linear regression, logistic regression, decision trees, SVM, etc.: ["Data Science", "Machine Learning Engineering"].
    * Data labeling/annotation, both manual and with the help of LLMs: ["Data Science", "Machine Learning Engineering"].
    * Traditional NLP techniques like TF-IDF, BM25, bag-of-words: ["Data Science", "Machine Learning Engineering"].
    * Anything related to deep learning, including neural networks, modern computer vision, modern NLP, large language models, and AI: ["Data Science", "Machine Learning Engineering"].
    * Anything related to retrieval-augmented generation (RAG), like vector indexing and vector search: ["Data Science", "Machine Learning Engineering"].
    * Anything related to information retrieval and search engines: ["Data Science", "Machine Learning Engineering"].
    * Anything related to deployment, operations, and scaling, including DevOps, CI/CD, cloud computing, computer networking, distributed computing, and containerization: ["Infrastructure and Operations"].
    * Anything related to MLOps, model deployment, and model monitoring: ["Infrastructure and Operations", "Machine Learning Engineering"].

    Text chunk: ```{{doc.content}}```
    """

# V13
prompt_template_tagging = """
    I am doing a content tagging project. The goal of the project is to tag teaching materials.

    I will give you a text chunk, delimited in triple backticks, and you will identify all applicable tags based on the text chunk. You can only pick tags from this list: Data Analysis, Data Engineering, Data Science, Infrastructure and Operations, Machine Learning Engineering.
    
    Please format the tags as a Python list of strings. Respond only with the list, no other text, e.g. ["Machine Learning Engineering", "Data Science"].
    
    If none of the tags apply, respond with an empty list, [].

    Here are my guidelines for how to decide which tags to choose. Carefully consider all guidelines before choosing the tags.
    * ["Data Engineering", "Data Science"]: Anything related to processing data before modeling or analysis, e.g. collecting, ingesting, cleaning, normalizing, and moving data. Anything related to database design and data modeling.
    * ["Data Analysis", "Data Science"]: Very simple data analysis like descriptive, diagnostic, and exploratory analysis, and any kind of data visualization, like dashboards.
    * ["Data Analysis"]: Business intelligence and Excel tutorials.
    * ["Data Analysis", "Data Engineering", "Data Science", "Machine Learning Engineering"]: SQL tutorials.
    * ["Data Science"]: Inferential data analysis. Feature engineering. Data labeling/annotation, both manual and with the help of LLMs.
    * ["Data Science", "Machine Learning Engineering"]: Predictive and prescriptive data analysis. Statistical modeling techniques like linear regression, logistic regression, decision trees, SVM, etc. Traditional NLP techniques like TF-IDF, BM25, bag-of-words. Anything related to deep learning, including neural networks, modern computer vision, modern NLP, large language models, AI, information retrieval and search engines. Anything related to retrieval-augmented generation (RAG), like vector indexing and vector search.
    * ["Infrastructure and Operations"]: Anything related to deployment, operations, and scaling, including DevOps, CI/CD, cloud computing, computer networking, distributed computing, and containerization.
    * ["Infrastructure and Operations", "Machine Learning Engineering"]: Anything related to MLOps, model deployment, and model monitoring.

    Text chunk: ```{{doc.content}}```
    """

# V14 PPTX + include document title
prompt_template_tagging = """
    I am doing a content tagging project. The goal of the project is to tag teaching materials.

    I will give you a text chunk, along with the document name that the text chunk came from, delimited in triple backticks, and you will identify all applicable tags based on the text chunk. You can only pick tags from this list: Data Analysis, Data Engineering, Data Science, Infrastructure and Operations, Machine Learning Engineering.
    
    Please format the tags as a Python list of strings. Respond only with the list, no other text, e.g. ["Machine Learning Engineering", "Data Science"].
    
    If none of the tags apply, respond with an empty list, [].

    Here are my guidelines for how to decide which tags to choose. I will give a topic followed by the recommended tags.
    * Anything related to processing data before modeling or analysis, e.g. collecting, ingesting, cleaning, normalizing, and moving data: ["Data Engineering", "Data Science"].
    * Anything related to database design and data modeling: ["Data Engineering", "Data Science"].
    * Very simple data analysis like descriptive, diagnostic, and exploratory analysis, and data visualization: ["Data Analysis", "Data Science"].
    * Business intelligence (commonly abbreviated as BI) and Excel tutorials: ["Data Analysis"].
    * Inferential data analysis: ["Data Science"].
    * Feature engineering: ["Data Science"].
    * Predictive and prescriptive data analysis: ["Data Science", "Machine Learning Engineering"].
    * Statistical modeling techniques like linear regression, logistic regression, decision trees, SVM, etc.: ["Data Science", "Machine Learning Engineering"].
    * Data labeling/annotation, both manual and with the help of LLMs: ["Data Science", "Machine Learning Engineering"].
    * Traditional NLP techniques like TF-IDF, BM25, bag-of-words: ["Data Science", "Machine Learning Engineering"].
    * Anything related to deep learning, including neural networks, modern computer vision, modern NLP, large language models, and AI: ["Data Science", "Machine Learning Engineering"].
    * Anything related to retrieval-augmented generation (RAG), like vector indexing and vector search: ["Data Science", "Machine Learning Engineering"].
    * Anything related to information retrieval and search engines: ["Data Science", "Machine Learning Engineering"].
    * Anything related to deployment, operations, and scaling, including DevOps, CI/CD, cloud computing, computer networking, distributed computing, and containerization: ["Infrastructure and Operations"].
    * Anything related to MLOps, model deployment, and model monitoring: ["Infrastructure and Operations", "Machine Learning Engineering"].

    ```Text chunk: {{doc.content}}\n\nDocument name: {{doc.meta["metadata"]["title"]}}```
    """